In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType

import reverse_geocoder as rg
from datetime import date, timedelta

StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 7, Finished, Available, Finished, False)

In [4]:
#remove all this before rnning data factory pipeline

start_date = date.today()-timedelta(7)

StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 8, Finished, Available, Finished, False)

In [5]:
df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)

StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 9, Finished, Available, Finished, False)

In [6]:
#CREATE A FUNCTION

def get_country_code(lat,lon):
    '''
    retrive the contry code for given latitude and longitude
    Parameter:
    lat (float or str) = latitude
    lon (float or str) = longitude

    Returns:
    str = contry code of the location using Reverse Geocode API

    Example:
    >>> get_country_code(19.233,2.9888)
    "FR"
'''
    coordinates = (float(lat),float(lon))
    return rg.search(coordinates)[0].get('cc')


StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 10, Finished, Available, Finished, False)

In [7]:
#register the UDF so that we can use it in Spark dataframe
get_country_code_udf = udf(get_country_code, StringType())


StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 11, Finished, Available, Finished, False)

In [8]:
# adding country code and city attributes
df_with_location =\
                    df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 12, Finished, Available, Finished, False)

In [9]:
# adding classification / Significance

df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class',
                                when(col("sig") < 100 , "Low") .\
                                when((col("sig") >= 100) & (col("sig") < 500) , "Moderate").\
                                otherwise("High")
                                )

StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 13, Finished, Available, Finished, False)

In [11]:
#appending data to gold Table
df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold')


StatementMeta(, 7dae5cda-1b63-446e-8698-3298ddd5eb3b, 15, Finished, Available, Finished, False)